# WaveletV3AELG Successive-Halving Analysis

Loads per-dataset WaveletV3AELG search results (TrendAELG + WaveletV3AELG stacks)
and emits a markdown report.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 80)

ROOT = Path('experiments')
if not ROOT.exists():
    ROOT = Path('.')

RESULTS = ROOT / 'results'
REPORT_PATH = ROOT / 'analysis_reports' / 'wavelet_v3aelg_study_analysis.md'

DATASET_ORDER = ['m4', 'tourism', 'traffic', 'weather']
DATASET_LABELS = {
    'm4': 'M4-Yearly',
    'tourism': 'Tourism-Yearly',
    'traffic': 'Traffic-96',
    'weather': 'Weather-96',
}

STUDY_VARIANTS = {
    'trendaelg': {
        'label': 'TrendAELG',
        'search_csvs': {ds: RESULTS / ds / 'wavelet_v3aelg_trendaelg_study_results.csv' for ds in DATASET_ORDER},
        'cross_csv': RESULTS / 'wavelet_v3aelg_trendaelg_cross_dataset_results.csv',
    },
}

In [2]:
def load_csv(path: Path):
    if not path.exists():
        return None
    df = pd.read_csv(path)
    for c in ['search_round', 'best_val_loss', 'trend_thetas_dim_cfg', 'latent_dim_cfg', 'basis_dim']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

# Load all data
all_data = {}
for vkey, vinfo in STUDY_VARIANTS.items():
    search_dfs = {ds: load_csv(path) for ds, path in vinfo['search_csvs'].items()}
    cross_df = load_csv(vinfo['cross_csv'])
    all_data[vkey] = {'search_dfs': search_dfs, 'cross_df': cross_df, 'label': vinfo['label']}

    print(f'--- {vinfo["label"]} ---')
    for ds in DATASET_ORDER:
        df = search_dfs[ds]
        if df is None:
            print(f'  {ds}: missing')
        else:
            print(f'  {ds}: rows={len(df)} configs={df["config_name"].nunique()} rounds={sorted(df["search_round"].dropna().astype(int).unique().tolist())}')
    print(f'  cross rows: {0 if cross_df is None else len(cross_df)}')

--- TrendAELG ---
  m4: rows=69 configs=23 rounds=[1]
  tourism: rows=762 configs=112 rounds=[1, 2, 3]
  traffic: rows=2 configs=1 rounds=[1]
  weather: rows=24 configs=8 rounds=[1]
  cross rows: 0


In [3]:
report = []
report.append('# WaveletV3AELG Study Analysis')
report.append('')

for vkey, vdata in all_data.items():
    label = vdata['label']
    search_dfs = vdata['search_dfs']
    report.append(f'# Trend Block: {label}')
    report.append('')

    for ds in DATASET_ORDER:
        df = search_dfs[ds]
        report.append(f'## {label} — Dataset: {DATASET_LABELS[ds]}')
        report.append('')
        if df is None or df.empty:
            report.append(f'- Missing CSV: `{STUDY_VARIANTS[vkey]["search_csvs"][ds]}`')
            report.append('')
            continue

        report.append(f'- Rows: {len(df)}')
        report.append(f'- Unique configs: {df["config_name"].nunique()}')
        report.append('')

        # Successive-halving funnel
        report.append('### Successive-Halving Funnel')
        rounds = sorted(df['search_round'].dropna().astype(int).unique().tolist())
        funnel_rows = []
        prev = None
        for r in rounds:
            rdf = df[df['search_round'].astype('Int64') == r]
            n_cfg = rdf['config_name'].nunique()
            keep = '-' if prev is None else f'{n_cfg}/{prev} ({n_cfg/max(prev,1):.0%})'
            best = pd.to_numeric(rdf['best_val_loss'], errors='coerce').median()
            funnel_rows.append({'Round': r, 'Configs': n_cfg, 'Rows': len(rdf), 'Median best_val_loss': best, 'Kept': keep})
            prev = n_cfg
        report.append(pd.DataFrame(funnel_rows).to_markdown(index=False, floatfmt='.4f'))
        report.append('')

        # Round 3 leaderboard
        report.append('### Round 3 Leaderboard (Top 10)')
        r3 = df[df['search_round'].astype('Int64') == 3].copy()
        if r3.empty:
            report.append('Round 3 not available.')
            report.append('')
        else:
            grp = (
                r3.groupby(['config_name', 'wavelet_family', 'bd_label', 'trend_thetas_dim_cfg', 'latent_dim_cfg'], dropna=False)
                .agg(median_best_val_loss=('best_val_loss', 'median'), std_best_val_loss=('best_val_loss', 'std'), n=('best_val_loss', 'count'))
                .sort_values('median_best_val_loss')
                .head(10)
                .reset_index()
            )
            report.append(grp.to_markdown(index=False, floatfmt='.4f'))
            report.append('')

        # Round 1 marginals
        report.append('### Round 1 Hyperparameter Marginals (median best_val_loss)')
        r1 = df[df['search_round'].astype('Int64') == 1].copy()
        for col in ['wavelet_family', 'bd_label', 'trend_thetas_dim_cfg', 'latent_dim_cfg']:
            if col not in r1.columns:
                continue
            mg = (
                r1.groupby(col)
                .agg(median_best_val_loss=('best_val_loss', 'median'), mean_best_val_loss=('best_val_loss', 'mean'), n=('best_val_loss', 'count'))
                .sort_values('median_best_val_loss')
                .reset_index()
            )
            report.append(f'#### {col}')
            report.append(mg.to_markdown(index=False, floatfmt='.4f'))
            report.append('')

In [4]:
# Per-dataset summary: best config and overall median
report.append('# Per-Dataset Summary')
report.append('')

for ds in DATASET_ORDER:
    vdata = all_data.get('trendaelg')
    if vdata is None:
        continue
    df = vdata['search_dfs'][ds]
    report.append(f'## {DATASET_LABELS[ds]}')
    report.append('')
    if df is None or df.empty:
        report.append('- No data available.')
        report.append('')
        continue

    r3 = df[df['search_round'].astype('Int64') == 3].copy()
    if r3.empty:
        report.append('- Round 3 not available.')
        report.append('')
        continue

    overall_median = r3['best_val_loss'].median()
    best_cfg = (
        r3.groupby('config_name')
        .agg(median_bvl=('best_val_loss', 'median'), n=('best_val_loss', 'count'))
        .sort_values('median_bvl')
        .iloc[0]
    )
    report.append(f'- Overall round-3 median best_val_loss: {overall_median:.4f}')
    report.append(f'- Best config: **{best_cfg.name}** (median={best_cfg["median_bvl"]:.4f}, n={int(best_cfg["n"])})')
    report.append('')

In [5]:
# Cross-dataset robustness
report.append('# Cross-Dataset Robustness')
report.append('')

for vkey, vdata in all_data.items():
    label = vdata['label']
    cross_df = vdata['cross_df']
    report.append(f'## {label}')
    report.append('')
    if cross_df is None or cross_df.empty:
        report.append(f'- Missing or empty cross CSV: `{STUDY_VARIANTS[vkey]["cross_csv"]}`')
        report.append('')
        continue
    cdf = cross_df.copy()
    metric = 'best_val_loss'
    if metric not in cdf.columns and 'final_val_loss' in cdf.columns:
        metric = 'final_val_loss'
    board = (
        cdf.groupby(['canonical_config_id', 'source_datasets'], dropna=False)
        .agg(mean_metric=(metric, 'mean'), std_metric=(metric, 'std'), n=(metric, 'count'))
        .sort_values('mean_metric')
        .reset_index()
    )
    report.append(board.to_markdown(index=False, floatfmt='.4f'))
    report.append('')
    report.append(f'- Total cross rows: {len(cdf)}')
    report.append('')

In [6]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text('\n'.join(report), encoding='utf-8')
print(f'Wrote report to: {REPORT_PATH}')
print('\n'.join(report[:30]))

Wrote report to: experiments/analysis_reports/wavelet_v3aelg_study_analysis.md
# WaveletV3AELG Study Analysis

# Trend Block: TrendAELG

## TrendAELG — Dataset: M4-Yearly

- Rows: 69
- Unique configs: 23

### Successive-Halving Funnel
|   Round |   Configs |   Rows |   Median best_val_loss | Kept   |
|--------:|----------:|-------:|-----------------------:|:-------|
|       1 |        23 |     69 |                16.9083 | -      |

### Round 3 Leaderboard (Top 10)
Round 3 not available.

### Round 1 Hyperparameter Marginals (median best_val_loss)
#### wavelet_family
| wavelet_family    |   median_best_val_loss |   mean_best_val_loss |   n |
|:------------------|-----------------------:|---------------------:|----:|
| DB2WaveletV3AELG  |                16.5923 |              16.9414 |  24 |
| HaarWaveletV3AELG |                17.1080 |              17.0307 |  24 |
| DB3WaveletV3AELG  |                17.5004 |              26.0462 |  21 |

#### bd_label
| bd_label   |   median_best_va